# Notebook 03_02 — Feature Engineering: LDA

**Supervised** linear feature extraction — uses class labels `y_train` during fit (this is the
correct, supervisor-required use of LDA, unlike the previous unsupervised misuse).

**Hard constraint**: LDA can produce at most `n_classes - 1` discriminant components.
With 10 classes here, max components = 9. When `K > 9`, components are clipped to 9 and
the remaining columns are zero-padded so the feature matrix shape still matches `K`
(documented in the `actual_K` column of the results).

**Results saved incrementally to** `results/fe_lda_raw.csv` — resumable on crash.

In [ ]:
import sys; sys.path.insert(0, '/home/hanh/UAV_/notebook')
from common import *
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

print('Imports OK')

In [ ]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
N_CLASSES = d['n_classes']
LDA_MAX_COMPONENTS = N_CLASSES - 1

METHOD_NAME = 'LDA'
RESULTS_CSV = f'{RESULTS_DIR}fe_lda_raw.csv'

print(f'n_classes={N_CLASSES}  ->  LDA max components = {LDA_MAX_COMPONENTS}')

In [ ]:
def transform_fn(K):
    K_lda = min(K, LDA_MAX_COMPONENTS)
    lda = LDA(n_components=K_lda)
    Xtr = lda.fit_transform(X_train, y_train)   # SUPERVISED: uses labels
    Xva = lda.transform(X_val)
    Xte = lda.transform(X_test)

    if K_lda < K:
        pad = K - K_lda
        Xtr = np.hstack([Xtr, np.zeros((len(Xtr), pad))])
        Xva = np.hstack([Xva, np.zeros((len(Xva), pad))])
        Xte = np.hstack([Xte, np.zeros((len(Xte), pad))])
        print(f'  [LDA K={K}] clipped to {K_lda} components (= n_classes - 1), zero-padded to {K}')

    return Xtr, Xva, Xte, K_lda

In [ ]:
run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

In [ ]:
res_df = pd.read_csv(RESULTS_CSV)
print(res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std']).to_string())
print('\nactual_K used per K:')
print(res_df.groupby('K')['actual_K'].first())